# Data Visualization — Advanced Reference
> **Level:** Advanced | **Goal:** Create production-quality static and interactive charts

## Table of Contents
1. [Matplotlib — Advanced Customization](#matplotlib)
2. [Seaborn — Statistical Plots](#seaborn)
3. [Plotly — Interactive Visualizations](#plotly)
4. [Chart Selection Guide](#selection)
5. [Design Best Practices](#design)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

# ── Sample data ────────────────────────────────────────────────
n = 1000
df = pd.DataFrame({
    'revenue':  rng.lognormal(5, 0.7, n),
    'sessions': rng.poisson(8, n).astype(float),
    'age':      rng.integers(18, 70, n),
    'converted': rng.binomial(1, 0.12, n),
    'channel':  rng.choice(['Organic','Paid','Email','Social'], n, p=[0.4,0.3,0.2,0.1]),
    'month':    rng.choice(pd.date_range('2023-01', periods=12, freq='MS'), n),
    'region':   rng.choice(['North','South','East','West'], n),
})
print("Data ready. Shape:", df.shape)

---
## 1 · Matplotlib — Advanced Customization <a id='matplotlib'></a>

In [ ]:
# ── Professional theme setup ───────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2196F3','#FF5722','#4CAF50','#FF9800','#9C27B0','#00BCD4']

def style_axis(ax, title='', xlabel='', ylabel='', legend=True):
    """Apply consistent styling to any Axes object."""
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.spines[['top','right']].set_visible(False)
    if legend and ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False, fontsize=10)
    return ax

In [ ]:
# ── GridSpec: complex multi-panel layout ──────────────────────
fig = plt.figure(figsize=(15, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: Revenue distribution with KDE
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(df['revenue'], bins=50, density=True, color=COLORS[0], alpha=0.7, label='Revenue')
from scipy.stats import gaussian_kde
kde = gaussian_kde(df['revenue'])
x_range = np.linspace(df['revenue'].min(), df['revenue'].max(), 200)
ax1.plot(x_range, kde(x_range), color='red', lw=2, label='KDE')
ax1.axvline(df['revenue'].median(), color='orange', ls='--', lw=1.5, label=f'Median={df["revenue"].median():.0f}')
style_axis(ax1, 'Revenue Distribution', 'Revenue ($)', 'Density')

# Panel 2: Channel revenue box plots
ax2 = fig.add_subplot(gs[0, 1])
channel_data = [df[df['channel']==ch]['revenue'].values for ch in df['channel'].unique()]
bp = ax2.boxplot(channel_data, labels=df['channel'].unique(), patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
style_axis(ax2, 'Revenue by Channel', 'Channel', 'Revenue ($)', legend=False)

# Panel 3: Conversion rate by channel (bar + CI)
ax3 = fig.add_subplot(gs[0, 2])
conv_by_channel = df.groupby('channel')['converted'].agg(['mean','sem','count'])
conv_by_channel['ci95'] = 1.96 * conv_by_channel['sem']
bars = ax3.bar(conv_by_channel.index, conv_by_channel['mean'],
               color=COLORS[:len(conv_by_channel)], alpha=0.8,
               yerr=conv_by_channel['ci95'], capsize=5, error_kw={'elinewidth':1.5})
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.1%}'))
style_axis(ax3, 'Conversion Rate by Channel (95% CI)', 'Channel', 'Conversion Rate', legend=False)

# Panel 4: Scatter with regression
ax4 = fig.add_subplot(gs[1, 0])
sc = ax4.scatter(df['age'], df['revenue'], c=df['sessions'], cmap='viridis',
                 alpha=0.3, s=15)
plt.colorbar(sc, ax=ax4, label='Sessions')
z = np.polyfit(df['age'], df['revenue'], 1)
ax4.plot(sorted(df['age']), np.poly1d(z)(sorted(df['age'])), 'r--', lw=2)
style_axis(ax4, 'Age vs Revenue (colored by Sessions)', 'Age', 'Revenue ($)', legend=False)

# Panel 5: Monthly revenue trend
ax5 = fig.add_subplot(gs[1, 1])
monthly = df.groupby('month')['revenue'].agg(['sum','mean']).reset_index()
ax5.bar(range(len(monthly)), monthly['sum']/1000, color=COLORS[0], alpha=0.6, label='Total (K)')
ax5_twin = ax5.twinx()
ax5_twin.plot(range(len(monthly)), monthly['mean'], 'o-', color=COLORS[1], lw=2, label='Avg')
ax5.set_xticks(range(len(monthly)))
ax5.set_xticklabels([m.strftime('%b') for m in monthly['month']], rotation=45)
style_axis(ax5, 'Monthly Revenue', 'Month', 'Total Revenue (K$)')

# Panel 6: Heatmap — correlation
ax6 = fig.add_subplot(gs[1, 2])
corr = df[['revenue','sessions','age','converted']].corr()
im = ax6.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax6)
ax6.set_xticks(range(len(corr))); ax6.set_xticklabels(corr.columns, rotation=45)
ax6.set_yticks(range(len(corr))); ax6.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax6.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=9)
style_axis(ax6, 'Correlation Matrix', legend=False)

plt.suptitle('Revenue Analytics Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Dashboard saved to dashboard.png")

---
## 2 · Seaborn — Statistical Plots <a id='seaborn'></a>

In [ ]:
sns.set_theme(style='whitegrid', palette='husl')

# ── PairPlot: multi-variable overview ─────────────────────────
g = sns.pairplot(
    df[['revenue','sessions','age','channel']].sample(500),
    hue='channel',
    diag_kind='kde',
    plot_kws={'alpha': 0.5, 's': 15},
    corner=True
)
g.fig.suptitle('Pairplot: Revenue, Sessions, Age by Channel', y=1.01)
plt.show()

In [ ]:
# ── FacetGrid: distribution by group ──────────────────────────
g = sns.FacetGrid(df, col='channel', col_wrap=2, height=3.5, sharey=False)
g.map_dataframe(sns.histplot, x='revenue', bins=40, kde=True)
g.set_titles('{col_name}')
g.set_axis_labels('Revenue', 'Count')
g.figure.suptitle('Revenue Distribution per Channel', y=1.02)
plt.show()

In [ ]:
# ── Violin + Strip: show distribution AND individual points ───
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Violin
sns.violinplot(data=df, x='channel', y='revenue', hue='channel',
               inner='quartile', palette='husl', ax=axes[0], legend=False)
axes[0].set_title('Revenue Distribution (Violin)', fontweight='bold')

# Violin + stripplot overlay
sns.violinplot(data=df.sample(200), x='channel', y='sessions',
               inner=None, palette='pastel', ax=axes[1])
sns.stripplot(data=df.sample(200), x='channel', y='sessions',
              size=3, alpha=0.6, jitter=True, palette='dark:k', ax=axes[1])
axes[1].set_title('Sessions per Channel (Violin + Strip)', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap with clustering ─────────────────────────────────────
pivot = df.pivot_table(values='revenue', index='channel', columns='region', aggfunc='mean')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Standard heatmap
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label':'Avg Revenue'})
axes[0].set_title('Avg Revenue: Channel × Region', fontweight='bold')

# Diverging heatmap (vs grand mean)
grand_mean = pivot.mean().mean()
pivot_diff = pivot - grand_mean
sns.heatmap(pivot_diff, annot=True, fmt='.0f', cmap='RdBu_r', center=0, ax=axes[1])
axes[1].set_title(f'Deviation from Grand Mean ({grand_mean:.0f})', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 3 · Plotly — Interactive Visualizations <a id='plotly'></a>

In [ ]:
# ── Interactive scatter with hover info ────────────────────────
fig = px.scatter(
    df.sample(400, random_state=0),
    x='age', y='revenue',
    color='channel',
    size='sessions',
    hover_data=['converted','region'],
    trendline='ols',
    title='Revenue vs Age (bubble size = sessions)',
    labels={'revenue': 'Revenue ($)', 'age': 'Age'},
    template='plotly_white'
)
fig.update_traces(marker=dict(opacity=0.7, line=dict(width=0.5, color='white')))
fig.show()

In [ ]:
# ── Interactive multi-panel dashboard ─────────────────────────
monthly_agg = df.groupby('month').agg(
    total_revenue=('revenue','sum'),
    avg_revenue=('revenue','mean'),
    n_orders=('revenue','count'),
    conversion_rate=('converted','mean')
).reset_index()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Monthly Total Revenue','Avg Order Value','Order Volume','Conversion Rate'],
    specs=[[{'type':'bar'},{'type':'scatter'}],
           [{'type':'scatter'},{'type':'scatter'}]]
)

fig.add_trace(go.Bar(x=monthly_agg['month'], y=monthly_agg['total_revenue'],
                     name='Total Revenue', marker_color='royalblue'), row=1, col=1)
fig.add_trace(go.Scatter(x=monthly_agg['month'], y=monthly_agg['avg_revenue'],
                         mode='lines+markers', name='Avg Order Value',
                         line=dict(color='crimson', width=2)), row=1, col=2)
fig.add_trace(go.Scatter(x=monthly_agg['month'], y=monthly_agg['n_orders'],
                         mode='lines+markers', fill='tozeroy', name='Orders',
                         line=dict(color='seagreen')), row=2, col=1)
fig.add_trace(go.Scatter(x=monthly_agg['month'], y=monthly_agg['conversion_rate'],
                         mode='lines+markers', name='CVR',
                         line=dict(color='darkorange', width=2, dash='dot')), row=2, col=2)

fig.update_layout(height=600, title_text='<b>Monthly Performance Dashboard</b>',
                  template='plotly_white', showlegend=False)
fig.show()

In [ ]:
# ── Funnel chart ───────────────────────────────────────────────
funnel_data = pd.DataFrame({
    'stage': ['Visitors','Sessions','Sign-ups','Trials','Paid'],
    'users':  [100000, 45000, 12000, 3200, 850]
})
funnel_data['conversion'] = funnel_data['users'] / funnel_data['users'].shift(1)

fig = go.Figure(go.Funnel(
    y=funnel_data['stage'],
    x=funnel_data['users'],
    textinfo='value+percent previous',
    marker=dict(color=['#1976D2','#2196F3','#64B5F6','#BBDEFB','#E3F2FD'])
))
fig.update_layout(title='<b>User Acquisition Funnel</b>', template='plotly_white')
fig.show()

In [ ]:
# ── Animated time series ───────────────────────────────────────
channel_monthly = df.groupby(['month','channel'])['revenue'].sum().reset_index()
channel_monthly['month_str'] = channel_monthly['month'].dt.strftime('%Y-%m')

fig = px.bar(
    channel_monthly,
    x='channel', y='revenue',
    color='channel',
    animation_frame='month_str',
    range_y=[0, channel_monthly['revenue'].max() * 1.1],
    title='<b>Revenue by Channel Over Time</b>',
    template='plotly_white'
)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 600
fig.show()

---
## 4 · Chart Selection Guide <a id='selection'></a>

| Goal | Chart type | Notes |
|---|---|---|
| Compare categories | Bar chart | Sort by value unless categories have order |
| Distribution shape | Histogram + KDE | Log scale if skewed |
| Two distributions | Violin / Box + Strip | Violin shows shape; box shows quartiles |
| Relationship between 2 continuous | Scatter | Add trendline, color 3rd variable |
| Time trend | Line chart | Add confidence band for forecasts |
| Part-to-whole | Stacked bar / Pie | Avoid pie for > 5 slices |
| Matrix of values | Heatmap | Diverging colormap if centered at 0 |
| Multi-variable overview | Pairplot / Parallel coordinates | Sample to ≤ 1000 points |
| Funnel / sequential conversion | Funnel chart | Show drop-off % at each step |
| Geographic | Choropleth / Bubble map | Normalize by population |
| Hierarchical | Treemap / Sunburst | Good for part-of-whole nested |

## 5 · Design Best Practices <a id='design'></a>

### Color
- Use a **colorblind-safe palette**: `sns.color_palette('colorblind')` or `viridis`
- Sequential colormap for ordered data (low → high)
- Diverging colormap when there's a meaningful center (0, baseline)
- Max **5–7 distinct colors** in categorical charts

### Labels & Annotations
- Title: **what** the chart shows + key takeaway in one line
- Always label axes with units
- Annotate outliers and key data points directly on the chart
- Add reference lines (mean, target, benchmark)

### Layout
- Remove chart junk: top/right spines, excessive gridlines
- Data-ink ratio: every pixel should encode information
- Consistent font sizes across a dashboard
- White space makes charts easier to read

In [ ]:
# ── Colorblind-safe palette check ─────────────────────────────
palettes = ['colorblind', 'husl', 'tab10', 'Set2']
fig, axes = plt.subplots(1, len(palettes), figsize=(14, 1.5))
for ax, name in zip(axes, palettes):
    palette = sns.color_palette(name, n_colors=8)
    for i, color in enumerate(palette):
        ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=color))
    ax.set_xlim(0, len(palette))
    ax.set_ylim(0, 1)
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.suptitle('Seaborn Palettes', fontsize=12)
plt.show()